In [ ]:
#📦 Install Dependencies
!pip -q install pymupdf pinecone langchain langchain-community langchain-pinecone langchain-groq

In [ ]:
import os, time, uuid
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("groq_key")
os.environ["PINECONE_API_KEY"] = userdata.get("pinecone_api")

In [ ]:
from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])

In [ ]:
#📄 Upload Multiple PDFs
from google.colab import files

uploaded = files.upload()
pdf_paths = list(uploaded.keys())
print(f"Loaded {len(pdf_paths)} PDF(s):")
for path in pdf_paths:
    print(f"  - {path}")

Saving H1_2024.pdf to H1_2024.pdf
Saving H1_2025.pdf to H1_2025.pdf
Saving H1_2026.pdf to H1_2026.pdf
Saving H2_2024.pdf to H2_2024.pdf
Saving H2_2025.pdf to H2_2025.pdf
Saving H2_2026.pdf to H2_2026.pdf
Loaded 6 PDF(s):
  - H1_2024.pdf
  - H1_2025.pdf
  - H1_2026.pdf
  - H2_2024.pdf
  - H2_2025.pdf
  - H2_2026.pdf


In [ ]:
#📖 Extract Text from Multiple PDFs and Create Chunks
import fitz  # PyMuPDF
from langchain_core.documents import Document

def extract_text_pymupdf(path: str) -> str:
    """Extract text from a PDF file."""
    doc = fitz.open(path)
    return "\n".join(page.get_text("text") for page in doc)

def chunk_raw_text(text: str, chunk_size=1000, overlap=150, source="pdf"):
    """Split text into overlapping chunks."""
    chunks, cid = [], 0
    start, n = 0, len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(Document(
            page_content=text[start:end],
            metadata={"source": source, "chunk_id": cid}
        ))
        cid += 1
        if end == n:
            break
        start = max(0, end - overlap)

    return chunks

# Process all PDFs and build tracking dictionary
all_docs = []
file_chunk_map = {}

for pdf_path in pdf_paths:
    # Extract text
    raw_text = extract_text_pymupdf(pdf_path).strip()
    print(f"\n{pdf_path}:")
    print(f"  Characters extracted: {len(raw_text)}")

    # Chunk the text
    docs = chunk_raw_text(raw_text, source=pdf_path)
    print(f"  Total chunks: {len(docs)}")

    # Generate unique IDs for chunks from this file
    doc_hash = uuid.uuid4().hex
    chunk_ids = [
        f"{doc_hash}_chunk_{d.metadata['chunk_id']}"
        for d in docs
    ]

    # Store in tracking dictionary
    file_chunk_map[pdf_path] = chunk_ids

    # Update document metadata with the generated IDs
    for doc, chunk_id in zip(docs, chunk_ids):
        doc.metadata['unique_id'] = chunk_id

    # Add to master list
    all_docs.extend(docs)

print(f"\n✅ Total chunks across all files: {len(all_docs)}")
print(f"\n📊 File-to-Chunk Mapping:")
for filename, ids in file_chunk_map.items():
    print(f"  {filename}: {len(ids)} chunks")


H1_2024.pdf:
  Characters extracted: 54875
  Total chunks: 65

H1_2025.pdf:
  Characters extracted: 61526
  Total chunks: 73

H1_2026.pdf:
  Characters extracted: 61526
  Total chunks: 73

H2_2024.pdf:
  Characters extracted: 35605
  Total chunks: 42

H2_2025.pdf:
  Characters extracted: 61106
  Total chunks: 72

H2_2026.pdf:
  Characters extracted: 79079
  Total chunks: 93

✅ Total chunks across all files: 418

📊 File-to-Chunk Mapping:
  H1_2024.pdf: 65 chunks
  H1_2025.pdf: 73 chunks
  H1_2026.pdf: 73 chunks
  H2_2024.pdf: 42 chunks
  H2_2025.pdf: 72 chunks
  H2_2026.pdf: 93 chunks


In [ ]:
#technique of embedding
# sparce = is there a specific word and make it one dimesion and then for other word make a new dimension (more specific)
# dense = how complecated or general this so more general

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os, time

INDEX_NAME = "pdf-rag-demo"
EMBED_MODEL = "multilingual-e5-large"
EMBED_DIM = 1024

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        vector_type="dense",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Created Pinecone index:", INDEX_NAME)
else:
    print("Using existing Pinecone index:", INDEX_NAME)

while not pc.describe_index(INDEX_NAME)["status"]["ready"]:
    print("Waiting for Pinecone index to be ready...")
    time.sleep(2)

print("Pinecone index is ready ✅")

Using existing Pinecone index: pdf-rag-demo
Pinecone index is ready ✅


In [ ]:
# 🔢 Create Embedding Model (Pinecone Inference)
from langchain_pinecone import PineconeEmbeddings
import os

embeddings = PineconeEmbeddings(
    model="multilingual-e5-large",
    pinecone_api_key=os.environ["PINECONE_API_KEY"]
)

# 📦 Create Vector Store (LangChain + Pinecone)
from langchain_pinecone import PineconeVectorStore

NAMESPACE = "testing"

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)

# Extract the unique IDs we generated
ids = [doc.metadata['unique_id'] for doc in all_docs]

# Upsert all chunks with their unique IDs
vectorstore.add_documents(
    documents=all_docs,
    ids=ids
)

print(f"✅ Upserted {len(all_docs)} chunks into Pinecone namespace: {NAMESPACE}")
print(f"\n📋 File-Chunk Mapping Dictionary:")
print("file_chunk_map = {")
for filename, chunk_ids in file_chunk_map.items():
    print(f'    "{filename}": {chunk_ids[:3]}{"..." if len(chunk_ids) > 3 else ""},')
print("}")

✅ Upserted 418 chunks into Pinecone namespace: testing

📋 File-Chunk Mapping Dictionary:
file_chunk_map = {
    "H1_2024.pdf": ['87d82d60897d471ca740a0c62fb19b2b_chunk_0', '87d82d60897d471ca740a0c62fb19b2b_chunk_1', '87d82d60897d471ca740a0c62fb19b2b_chunk_2']...,
    "H1_2025.pdf": ['afc1f69af85b4f679c8075bc2f8a3691_chunk_0', 'afc1f69af85b4f679c8075bc2f8a3691_chunk_1', 'afc1f69af85b4f679c8075bc2f8a3691_chunk_2']...,
    "H1_2026.pdf": ['247d317b3115492caaab76508464b719_chunk_0', '247d317b3115492caaab76508464b719_chunk_1', '247d317b3115492caaab76508464b719_chunk_2']...,
    "H2_2024.pdf": ['f79bf9cdf62a4d219b2067438829688b_chunk_0', 'f79bf9cdf62a4d219b2067438829688b_chunk_1', 'f79bf9cdf62a4d219b2067438829688b_chunk_2']...,
    "H2_2025.pdf": ['7779c16f3a124848b1d7dd0cabe5b7df_chunk_0', '7779c16f3a124848b1d7dd0cabe5b7df_chunk_1', '7779c16f3a124848b1d7dd0cabe5b7df_chunk_2']...,
    "H2_2026.pdf": ['17486a20866c4db3bdb31906c54f2b56_chunk_0', '17486a20866c4db3bdb31906c54f2b56_chunk_1', '174

In [ ]:
file_chunk_map

{'H1_2024.pdf': ['87d82d60897d471ca740a0c62fb19b2b_chunk_0',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_1',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_2',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_3',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_4',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_5',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_6',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_7',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_8',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_9',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_10',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_11',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_12',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_13',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_14',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_15',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_16',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_17',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_18',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_19',
  '87d82d60897d471ca740a0c62fb19b2b_chunk_20',
  '87d82

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os, time

INDEX_NAME = "pdf-rag-demo"
EMBED_MODEL = "multilingual-e5-large"
EMBED_DIM = 1024

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing_indexes = [idx["name"] for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        vector_type="dense",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Created Pinecone index:", INDEX_NAME)
else:
    print("Using existing Pinecone index:", INDEX_NAME)


while not pc.describe_index(INDEX_NAME)["status"]["ready"]:
    print("Waiting for Pinecone index to be ready...")
    time.sleep(2)

print("Pinecone index is ready ✅")
""

Using existing Pinecone index: pdf-rag-demo
Pinecone index is ready ✅


''

In [ ]:
# 🔢 Create Embedding Model (Pinecone Inference)
from langchain_pinecone import PineconeEmbeddings
import os

embeddings = PineconeEmbeddings(
    model="multilingual-e5-large",
    pinecone_api_key=os.environ["PINECONE_API_KEY"]
)

# 📦 Create Vector Store (LangChain + Pinecone)
from langchain_pinecone import PineconeVectorStore

NAMESPACE = "testing"

vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE
)
import uuid
doc_hash = uuid.uuid4().hex  # e.g. 'a3f9c1e4d8b24f0c...'

ids = [
    f"{doc_hash}_chunk_{d.metadata['chunk_id']}"
    for d in docs
]


vectorstore.add_documents(
    documents=docs,
    ids=ids
)

print(f"Upserted {len(docs)} chunks into Pinecone namespace: {NAMESPACE}")

Upserted 93 chunks into Pinecone namespace: testing


In [ ]:
#🔍 Create Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}  # top 4 relevant chunks
)

#📝 RAG Prompt Template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Answer ONLY using the provided context from the PDFs. "
        "If the context is insufficient, say: 'I don't know based on the documents.'"
    ),
    MessagesPlaceholder(variable_name="history"),
    (
        "human",
        "Question: {question}\n\nContext:\n{context}"
    )
])

#Create the LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

In [ ]:
#🗂️ Initialize History
from langchain_community.chat_message_histories import ChatMessageHistory

history = ChatMessageHistory()

#🧱 Format Retrieved Context (Debug-Friendly)
def format_context(retrieved_docs):
    """
    Combines retrieved chunks into a single context string.
    Includes source file and chunk_id for transparency.
    """
    return "\n\n---\n\n".join(
        f"[source={d.metadata.get('source')}, chunk_id={d.metadata.get('chunk_id')}] {d.page_content}"
        for d in retrieved_docs
    )

In [ ]:
#🔄 Full RAG Flow
print("\n✅ RAG chatbot ready!")
print("Ask questions about the uploaded PDFs.")
print("Type 'exit' to stop.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in {"exit", "quit"}:
        break

    # 1️⃣ Retrieve relevant chunks
    retrieved_docs = retriever.invoke(f"query: {question}")

    # 2️⃣ Build context
    context = format_context(retrieved_docs)

    # 3️⃣ Create messages for the LLM
    messages = rag_prompt.format_messages(
        history=history.messages,
        question=question,
        context=context
    )

    # 4️⃣ Generate answer
    answer = llm.invoke(messages).content

    # 5️⃣ Save conversation history
    history.add_user_message(question)
    history.add_ai_message(answer)

    # 6️⃣ Display output with source tracking
    print("\nBot:", answer)
    print(
        "(Retrieved from sources:",
        list(set([d.metadata.get("source") for d in retrieved_docs])),
        ")"
    )
    print(
        "(Chunk IDs:",
        [d.metadata.get("chunk_id") for d in retrieved_docs],
        ")"
    )
    print()


✅ RAG chatbot ready!
Ask questions about the uploaded PDFs.
Type 'exit' to stop.

You: give summary

Bot: **Summary of the provided documents**

*H1_2024.pdf (chunk 11.0)*  
- Describes a mobile‑first health‑reporting system.  
- **Frontend**: React Native app with offline‑first architecture (AsyncStorage, background sync).  
- **Dashboard**: React web app using Recharts (charts) and Leaflet (maps).  
- **Backend**: Node.js/Express REST API with JWT authentication and role‑based access control.  
- **ML Pipeline**: Python microservice using scikit‑learn and TensorFlow for predictions.  
- **Databases**: PostgreSQL (structured data), MongoDB (health records), Redis (caching).  
- **Data flow**: Health worker submits a symptom report → stored locally → synced to API Gateway when online → API validates and stores in PostgreSQL/MongoDB → background jobs process the data.

*H2_2026.pdf (chunk 20.0)*  
- **Query flow**: User query → matched against FAISS vector DB → top 5 documents + query 

In [ ]:
import json
from typing import List, Dict
import os

In [ ]:
# The 12 Evaluation Criteria
EVALUATION_CRITERIA = [
    "Is the project scope, objective, and target usage context clearly specified?",
    "Is the problem statement precisely scoped with explicit assumptions?",
    "Does the implemented solution logically satisfy the problem requirements?",
    "Is the system architecture described (components, interfaces, data flow)?",
    "Are implemented features formally specified (inputs/outputs)?",
    "Is the technology stack version-pinned and justified?",
    "Are installation steps sufficient for deterministic reproduction?",
    "Are usage instructions defined with concrete examples?",
    "Are design trade-offs and failure modes documented?",
    "Are results validated using measurable metrics/benchmarks?",
    "Is the codebase structured for maintainability/extensibility?"
]

print(f"✅ Loaded {len(EVALUATION_CRITERIA)} evaluation criteria")

✅ Loaded 11 evaluation criteria


In [ ]:
def get_text_from_ids(chunk_ids: List[str]) -> str:
    """
    Retrieve text content from Pinecone using chunk IDs.

    OPTION 1 (MOCK - for testing):
    Use the mock implementation below

    OPTION 2 (PRODUCTION):
    Uncomment the production code to use real Pinecone retrieval
    """

    # ===== MOCK IMPLEMENTATION (for testing) =====
    import random
    random.seed(hash(str(chunk_ids)))

    mock_texts = {
        "id_1": "Team Alpha: AI code review system. Owner: John Doe. Clear architecture.",
        "id_2": "Tech stack: React 18.0, FastAPI 0.95, PostgreSQL 14. Well documented.",
        "id_3": "Installation via Docker. Usage examples provided. Test coverage 85%.",
    }

    retrieved_texts = [mock_texts.get(cid, f"Project documentation for {cid}") for cid in chunk_ids]
    return "\n\n".join(retrieved_texts)


print("✅ get_text_from_ids() function defined")

✅ get_text_from_ids() function defined


In [ ]:
def get_llm_rating(context: str, criteria: List[str]) -> Dict[str, int]:
    """
    Get LLM evaluation scores for each criterion.

    OPTION 1 (MOCK - for testing):
    Use the mock implementation below

    OPTION 2 (PRODUCTION):
    Uncomment the production code to use real LLM
    """

    import random
    random.seed(hash(context))

    scores = {}
    for i in range(1, len(criteria) + 1):
        scores[f"criterion_{i}"] = random.randint(5, 10)
    return scores

print("✅ get_llm_rating() function defined")

✅ get_llm_rating() function defined


In [ ]:
def evaluate_project(filename: str, chunk_ids: List[str]) -> Dict:
    """Evaluate a single project file."""

    print(f"\n{'='*80}")
    print(f"EVALUATING: {filename}")
    print(f"{'='*80}")

    # Step 1: Retrieve context
    print(f"\n🔍 Retrieving {len(chunk_ids)} chunks...")
    context = get_text_from_ids(chunk_ids)
    print(f"Retrieved {len(context)} characters")

    # Step 2: Get LLM ratings
    print(f"\nSending to LLM for evaluation...")
    scores = get_llm_rating(context, EVALUATION_CRITERIA)
    print(f"Received scores for {len(scores)} criteria")

    # Step 3: Generate report
    print(f"\nEVALUATION REPORT")
    print(f"{'-'*80}")

    total_score = 0
    for i, criterion in enumerate(EVALUATION_CRITERIA, 1):
        score = scores[f"criterion_{i}"]
        total_score += score
        print(f"{i:2d}. {criterion}")
        print(f"    Score: {score}/10")
        print()

    average_score = total_score / len(EVALUATION_CRITERIA)

    print(f"{'-'*80}")
    print(f"FINAL AVERAGE RATING: {average_score:.2f}/10")
    print(f"{'='*80}\n")

    return {
        "filename": filename,
        "scores": scores,
        "average_rating": average_score,
        "total_score": total_score
    }

print("evaluate_project() function defined")

evaluate_project() function defined


In [ ]:
def evaluate_all_projects(file_chunk_map: Dict[str, List[str]]) -> List[Dict]:
    """Evaluate all projects in the file_chunk_map."""

    print("\n🚀 HACKATHON PROJECT EVALUATION SYSTEM")
    print(f"📁 Total projects: {len(file_chunk_map)}\n")

    all_results = []

    # Process each project sequentially
    for filename, chunk_ids in file_chunk_map.items():
        result = evaluate_project(filename, chunk_ids)
        all_results.append(result)

    # Summary report
    print("\n" + "="*80)
    print("EVALUATION SUMMARY")
    print("="*80)

    sorted_results = sorted(all_results, key=lambda x: x["average_rating"], reverse=True)

    print(f"\n{'Rank':<6} {'Project':<40} {'Score':<10} {'Rating'}")
    print("-"*80)

    for rank, result in enumerate(sorted_results, 1):
        avg_score = result["average_rating"]

        if avg_score >= 9:
            badge = "EXCELLENT"
        elif avg_score >= 7.5:
            badge = "VERY GOOD"
        elif avg_score >= 6:
            badge = "GOOD"
        else:
            badge = "NEEDS WORK"

        print(f"{rank:<6} {result['filename']:<40} {avg_score:>5.2f}/10   {badge}")

    print("="*80)

    return all_results

print("✅ evaluate_all_projects() function defined")

✅ evaluate_all_projects() function defined


In [ ]:
# REPLACE THIS with your actual file_chunk_map from the RAG notebook
# This is the dictionary you created during PDF processing

# Run evaluation
results = evaluate_all_projects(file_chunk_map)


🚀 HACKATHON PROJECT EVALUATION SYSTEM
📁 Total projects: 6


EVALUATING: H1_2024.pdf

🔍 Retrieving 65 chunks...
Retrieved 4473 characters

Sending to LLM for evaluation...
Received scores for 11 criteria

EVALUATION REPORT
--------------------------------------------------------------------------------
 1. Is the project scope, objective, and target usage context clearly specified?
    Score: 7/10

 2. Is the problem statement precisely scoped with explicit assumptions?
    Score: 10/10

 3. Does the implemented solution logically satisfy the problem requirements?
    Score: 5/10

 4. Is the system architecture described (components, interfaces, data flow)?
    Score: 6/10

 5. Are implemented features formally specified (inputs/outputs)?
    Score: 9/10

 6. Is the technology stack version-pinned and justified?
    Score: 6/10

 7. Are installation steps sufficient for deterministic reproduction?
    Score: 10/10

 8. Are usage instructions defined with concrete examples?
    Score: 1